# Batch Box Segmentation — Grounding DINO + SAM2

**Top-down pipeline:** Grounding DINO detects boxes by text prompt → SAM2 Predictor refines masks.

**Why faster than AutoMaskGenerator?**
- AutoMaskGenerator blind-scans the whole image (144+ point grid × crop refinement)
- GDINO does one Swin-T forward (<1s) to locate all boxes, then SAM2 Predictor only runs mask decode (~0.05s/box)
- Net: ~2.5s/image vs ~30s

**Pipeline:**
1. Grounding DINO (Swin-T) — detect boxes via text prompt
2. Boundary filter — skip boxes touching frame edges
3. SAM2 Predictor — bbox-prompt mask refinement
4. Morphological clean-up → save RGBA per box

## 0. One-Time Setup

Run this cell **once** per instance to install dependencies and download models.
After that you can skip to Section 1.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# One-time setup — run once, then restart kernel & skip to §1
# ═══════════════════════════════════════════════════════════════

# 1. Install Grounding DINO
!pip install groundingdino-py -q

# 2. Download Swin-T weights
!mkdir -p /workspace/checkpoints
!wget -nc -P /workspace/checkpoints \
  https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth

# 3. BERT tokenizer (HF mirror for China users)
import os
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

from transformers import AutoTokenizer
AutoTokenizer.from_pretrained("bert-base-uncased")

# 4. Verify
from groundingdino.util.inference import load_model
print("\nGrounding DINO setup complete.")
print("→ Restart kernel, then run from Section 1.")

## 1. Configuration & Hyperparameters

In [ ]:
import os
import gc
import time
import numpy as np
import torch
from PIL import Image
from skimage.morphology import opening, closing, disk

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from groundingdino.util.inference import load_model, predict as gdino_predict

print("Imports ready.")

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
INPUT_DIR  = "/workspace/sam/input"
OUTPUT_DIR = "/workspace/sam/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── SAM2 ──
SAM2_CHECKPOINT = "/workspace/segment-anything-2/checkpoints/sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"

# ── Grounding DINO ──
GDINO_CONFIG = "groundingdino/config/GroundingDINO_SwinT_OGC.py"
GDINO_WEIGHTS = "/workspace/checkpoints/groundingdino_swint_ogc.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ------------------------------------------------------------------
# Image resizing
# ------------------------------------------------------------------
MAX_IMAGE_SIZE = 1024

# ------------------------------------------------------------------
# Grounding DINO detection parameters
# ------------------------------------------------------------------
# Text prompt — what to look for. Be specific but cover variants.
TEXT_PROMPT = "cardboard box. package. parcel. carton."

# Detection thresholds
# Lower → higher recall (catch more boxes), higher → higher precision
BOX_THRESHOLD = 0.30     # Ignore boxes with confidence below this
TEXT_THRESHOLD = 0.25    # Ignore text matches below this

# ------------------------------------------------------------------
# Box boundary filter
# ------------------------------------------------------------------
# Skip boxes that touch the image edge (partial boxes)
BOUNDARY_MARGIN = 8

# ------------------------------------------------------------------
# SAM2 Predictor
# ------------------------------------------------------------------
REFINE_MIN_SCORE = 0.80   # Minimum SAM2 IoU score to accept a mask

# ------------------------------------------------------------------
# Post-processing
# ------------------------------------------------------------------
MORPH_OPEN_RADIUS = 2
MORPH_CLOSE_RADIUS = 3

print("Configuration loaded.")

## 2. Helper Functions

In [ ]:
def is_fully_in_frame(bbox, img_h: int, img_w: int, margin: int = 8):
    """Return True if bbox is completely inside the frame (not touching edges)."""
    x1, y1, x2, y2 = bbox
    return (x1 >= margin and y1 >= margin and
            x2 < img_w - margin and y2 < img_h - margin)


def morphological_cleanup(mask, open_r=2, close_r=3):
    """Remove noise and fill small holes."""
    if open_r > 0:
        mask = opening(mask, disk(open_r))
    if close_r > 0:
        mask = closing(mask, disk(close_r))
    return mask


def save_box_rgba(image: np.ndarray, mask: np.ndarray, out_path: str):
    """Save box region with transparent background."""
    h, w = image.shape[:2]
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[:, :, :3] = image
    rgba[:, :, 3] = (mask.astype(np.uint8)) * 255
    Image.fromarray(rgba, "RGBA").save(out_path)


def save_overlay(image: np.ndarray, boxes_info, out_path: str):
    """Draw all bounding boxes onto the source image."""
    from PIL import ImageDraw
    pil_img = Image.fromarray(image)
    draw = ImageDraw.Draw(pil_img)
    for idx, (_, bbox, _) in enumerate(boxes_info):
        x1, y1, x2, y2 = bbox
        draw.rectangle([x1, y1, x2, y2], outline="#00FF00", width=3)
        draw.text((x1 + 4, y1 + 4), str(idx), fill="#00FF00")
    pil_img.save(out_path)

print("Helper functions defined.")

## 3. Load Models

Loads Grounding DINO and SAM2 in sequence. Both can stay in VRAM (>24GB).
If you have a 12-16GB GPU we'll unload GDINO after detection.

In [ ]:
# ── Load Grounding DINO ──
t_load = time.time()
gdino_model = load_model(GDINO_CONFIG, GDINO_WEIGHTS)
print(f"Grounding DINO Swin-T loaded in {time.time() - t_load:.1f}s")

# ── Load SAM2 Predictor (no AutoMaskGenerator needed) ──
t_load = time.time()
sam2 = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=device)
predictor = SAM2ImagePredictor(sam_model=sam2)
print(f"SAM2 Predictor loaded in {time.time() - t_load:.1f}s")

# Quick VRAM check
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    print(f"GPU memory: {allocated:.1f} GiB allocated, {reserved:.1f} GiB reserved")

## 4. Detect & Refine — Main Loop

Per image:
1. GDINO forward pass → (bboxes, confidences, class labels)
2. Filter out boundary-touching boxes
3. SAM2 Predictor bbox-prompt refinement
4. Morphological cleanup + save

In [ ]:
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
image_files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(image_extensions)
])

if not image_files:
    print(f"No images found in {INPUT_DIR}")
else:
    print(f"Found {len(image_files)} image(s).\n")

for img_file in image_files:
    t0 = time.time()
    img_path = os.path.join(INPUT_DIR, img_file)
    base_name = os.path.splitext(img_file)[0]
    out_subdir = os.path.join(OUTPUT_DIR, base_name)
    os.makedirs(out_subdir, exist_ok=True)

    # --------------------------------------------------------------
    # Load & resize
    # --------------------------------------------------------------
    pil_img = Image.open(img_path).convert("RGB")
    if max(pil_img.size) > MAX_IMAGE_SIZE:
        pil_img.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), Image.Resampling.LANCZOS)
    image = np.array(pil_img)
    img_h, img_w = image.shape[:2]
    print(f"[{base_name}] {img_w}x{img_h} — loaded.")

    # --------------------------------------------------------------
    # Stage 1: Grounding DINO detection
    # --------------------------------------------------------------
    t_gd = time.time()
    torch.cuda.empty_cache()
    gc.collect()

    with torch.no_grad():
        boxes, logits, phrases = gdino_predict(
            model=gdino_model,
            image=image,
            caption=TEXT_PROMPT,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
        )
    t_gd = time.time() - t_gd
    print(f"  Stage 1 GDINO: {len(boxes)} boxes detected ({t_gd:.1f}s) "
          f"[prompt: '{TEXT_PROMPT}']")
    if len(boxes) > 0:
        for i, (b, l, p) in enumerate(zip(boxes.tolist(), logits.tolist(), phrases)):
            print(f"    [{i}] {p}: conf={l:.2f} @ {[round(v, 1) for v in b]}")

    if len(boxes) == 0:
        print(f"  No boxes detected — skipping.\n")
        continue

    # --------------------------------------------------------------
    # Stage 2: Boundary filter (skip boxes touching frame edge)
    # --------------------------------------------------------------
    valid_bboxes = []
    for bbox in boxes.tolist():
        x1, y1, x2, y2 = map(int, bbox)
        if is_fully_in_frame((x1, y1, x2, y2), img_h, img_w, margin=BOUNDARY_MARGIN):
            valid_bboxes.append((x1, y1, x2, y2))
        else:
            print(f"    ⛔ skipped box @ [{x1},{y1},{x2},{y2}] — touches frame edge")
    print(f"  Stage 2 boundary filter: {len(valid_bboxes)}/{len(boxes)} boxes are fully in frame.")

    if not valid_bboxes:
        print(f"  No valid boxes — skipping.\n")
        continue

    # --------------------------------------------------------------
    # Stage 3: SAM2 Predictor refinement
    # --------------------------------------------------------------
    t_sam = time.time()
    predictor.set_image(image)

    results = []
    for i, bbox in enumerate(valid_bboxes):
        input_box = np.array(bbox)
        with torch.no_grad():
            masks_pred, scores_pred, _ = predictor.predict(
                point_coords=None,
                point_labels=None,
                box=input_box[None, :],
                multimask_output=True,
            )
        best_idx = int(np.argmax(scores_pred))
        best_mask = masks_pred[best_idx]
        best_score = float(scores_pred[best_idx])
        if best_score >= REFINE_MIN_SCORE:
            results.append((best_mask, bbox, best_score))
    t_sam = time.time() - t_sam
    print(f"  Stage 3 SAM2: {len(results)} masks refined ({len(results) - len(valid_bboxes) + len(valid_bboxes)}/{len(valid_bboxes)} passed score ≥{REFINE_MIN_SCORE}, {t_sam:.1f}s)")

    if not results:
        print(f"  No masks passed SAM2 refinement — skipping.\n")
        continue

    # --------------------------------------------------------------
    # Stage 4: Morphological cleanup + Save
    # --------------------------------------------------------------
    for i, (mask, bbox, score) in enumerate(results):
        mask = morphological_cleanup(
            mask,
            open_r=MORPH_OPEN_RADIUS,
            close_r=MORPH_CLOSE_RADIUS,
        )
        out_path = os.path.join(out_subdir, f"box_{i:03d}_score{score:.2f}.png")
        save_box_rgba(image, mask, out_path)

    overlay_path = os.path.join(out_subdir, "_overlay_boxes.png")
    save_overlay(image, results, overlay_path)

    elapsed = time.time() - t0
    print(f"  ✓ Saved {len(results)} box(es) to {out_subdir}/ ({elapsed:.1f}s total)\n")

    # Cleanup per-image GPU memory
    del results, valid_bboxes
    torch.cuda.empty_cache()
    gc.collect()

print("\n=== All images processed ===")
print(f"Output directory: {OUTPUT_DIR}")

## 5. Tuning Guide

### Detection

| Symptom | Fix | Parameter |
|---|---|---|
| Missing boxes | Lower thresholds | `BOX_THRESHOLD = 0.20`, `TEXT_THRESHOLD = 0.20` |
| False positives | Raise thresholds | `BOX_THRESHOLD = 0.40` |
| Missing small boxes | Add prompt keywords | `TEXT_PROMPT = "small box. cardboard box. package."` |
| Picking up wrong objects | Narrow prompt | `TEXT_PROMPT = "cardboard box."` |
| Edge boxes appear | Increase margin | `BOUNDARY_MARGIN = 15` |

### SAM2

| Symptom | Fix | Parameter |
|---|---|---|
| Masks too loose | Raise minimum score | `REFINE_MIN_SCORE = 0.90` |
| Masks too tight (missing box parts) | Lower minimum score | `REFINE_MIN_SCORE = 0.70` |
| Noisy mask edges | Increase opening | `MORPH_OPEN_RADIUS = 3` |
| Holes in masks | Increase closing | `MORPH_CLOSE_RADIUS = 5` |

### VRAM

| Symptom | Fix | Parameter |
|---|---|---|
| CUDA OOM on large images | Reduce resolution | `MAX_IMAGE_SIZE = 768` |
| Still OOM after resolution drop | Unload GDINO after detection | Add `del gdino_model; torch.cuda.empty_cache()` before SAM2 |